# Session 4: Evals and Reliability

**Great American Insurance Group — AI Developer Training**

In Sessions 1–3 you built an underwriting agent: extraction, structured outputs, RAG, tools, and an agent loop that produces a grounded appetite decision. Every output so far has been judged by reading it and deciding "that looks right." Today you'll replace that gut check with a repeatable evaluation suite — deterministic checks, an LLM-as-a-judge rubric, and lightweight observability — so you can tell, in a rerunnable way, whether a change to the system made things better or worse.

Throughout the notebook you'll see:
- 📝 **Exercise** / **Try It Yourself** — write or modify something yourself
- 💬 **Reflection** — a question to think about
- 💡 **Key Takeaway** — the main point to walk away with

Same rule as Sessions 1–3: all LLM calls go through `get_client().generate(...)` / `get_client().generate_structured(...)` / `get_client().generate_with_tools(...)` — never a raw `boto3`/`openai` SDK call.

This session reuses the same 4 synthetic underwriting PDFs from Session 2 (`../session-2-rag-retrieval/docs/`) and the agent built in Session 3 — no new documents, no rebuilt agent. You're picking this up on a different day with a fresh kernel, so Section 1 below rebuilds everything needed from scratch, exactly the way Session 3 rebuilt Session 2's chunking code instead of assuming that notebook was still open.

## Section 1 — Recap: reload the underwriting agent

**Goal:** Get back to a working agent as fast as possible so we can spend the session's time on evaluation, not rebuilding.

Everything in this section is a straight copy of Session 2's retrieval pipeline and Session 3's tools/agent — same PDFs, same chunking, same tool definitions, same agent loop. Nothing here is new teaching content; run these cells top to bottom and move on.

In [ ]:
from shared.llm import get_client

client = get_client()

In [ ]:
from pathlib import Path

from pypdf import PdfReader

# Same 4 PDFs as Sessions 2-3 - read straight from Session 2's docs/ folder rather
# than duplicating them here.
DOCS_DIR = Path("../session-2-rag-retrieval/docs")

DOC_IDS = ["GL-GUIDELINES", "PROPERTY-APPETITE", "CYBER-EXCLUSIONS", "WORKCOMP-CLASSIFICATION"]

def load_pdf_doc(path: Path) -> dict:
    """Extract title + body text from one of our underwriting-manual PDFs."""
    reader = PdfReader(path)
    raw_text = "\n".join(page.extract_text() for page in reader.pages)
    lines = [line.strip() for line in raw_text.splitlines() if line.strip()]

    title = lines[1]
    body = " ".join(lines[3:])
    return {"title": title, "text": body}

UNDERWRITING_DOCS = {}
for doc_id in DOC_IDS:
    filename = doc_id.lower() + ".pdf"
    UNDERWRITING_DOCS[doc_id] = load_pdf_doc(DOCS_DIR / filename)

print(f"Loaded {len(UNDERWRITING_DOCS)} documents.")

In [ ]:
import math
import re

def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    return dot / (norm_a * norm_b)

def semantic_chunk(text: str, percentile: float = 40) -> list[str]:
    """Group sentences into chunks based on embedding similarity, splitting where
    similarity drops below the given percentile of this document's own
    sentence-to-sentence similarity distribution - not a fixed size or a fixed
    absolute similarity value.
    """
    sentences = [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s]
    if len(sentences) <= 1:
        return sentences

    embeddings = client.embed(sentences)
    similarities = [
        cosine_similarity(embeddings[i - 1], embeddings[i])
        for i in range(1, len(sentences))
    ]

    threshold = sorted(similarities)[int(len(similarities) * percentile / 100)]

    chunks = []
    current_chunk = [sentences[0]]
    for i, similarity in enumerate(similarities, start=1):
        if similarity >= threshold:
            current_chunk.append(sentences[i])
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]
    chunks.append(" ".join(current_chunk))
    return chunks

In [ ]:
import chromadb

ALL_CHUNKS = []
for doc_id, doc in UNDERWRITING_DOCS.items():
    for i, chunk in enumerate(semantic_chunk(doc["text"])):
        ALL_CHUNKS.append({"text": chunk, "source": doc_id, "chunk_index": i})

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("underwriting_docs")

chunk_texts = [c["text"] for c in ALL_CHUNKS]
chunk_embeddings = client.embed(chunk_texts)

collection.add(
    ids=[f"{c['source']}-{c['chunk_index']}" for c in ALL_CHUNKS],
    embeddings=chunk_embeddings,
    documents=chunk_texts,
    metadatas=[{"source": c["source"], "chunk_index": c["chunk_index"]} for c in ALL_CHUNKS],
)

print("Collection count:", collection.count())

In [ ]:
def retrieve(query: str, k: int = 3):
    query_embedding = client.embed([query])[0]
    return collection.query(query_embeddings=[query_embedding], n_results=k)

def search_documents(query: str, k: int = 3) -> list[dict]:
    """Search the underwriting guideline manuals for text relevant to `query`."""
    results = retrieve(query, k=k)

    hits = []
    for text, metadata, distance in zip(
        results["documents"][0], results["metadatas"][0], results["distances"][0]
    ):
        hits.append(
            {
                "source": metadata["source"],
                "chunk_index": metadata["chunk_index"],
                "text": text,
                "distance": distance,
            }
        )

    return hits

In [ ]:
from pydantic import BaseModel, Field

class SubmissionExtraction(BaseModel):
    """Extraction schema for underwriting submissions (same as Sessions 1 and 3)."""
    business_name: str
    industry: str
    locations: list[str]
    requested_limits: str | None = None
    notable_risks: list[str]
    missing_information: list[str]

EXTRACTION_SYSTEM_PROMPT = (
    "Extract underwriting information from the submission text. "
    "For the 'requested_limits' field, if no specific amount is mentioned, set it to null. "
    "For 'notable_risks' and 'missing_information', use empty lists if there are none. "
    "Do not invent values for missing fields."
)

def extract_submission_information(submission_text: str) -> dict:
    """Pull structured facts out of raw submission text."""
    result = client.generate_structured(
        submission_text, SubmissionExtraction, system_prompt=EXTRACTION_SYSTEM_PROMPT
    )
    return result.model_dump()

In [ ]:
def parse_dollar_amount(text: str | None) -> float | None:
    """Pull the first dollar figure out of a free-text limit string, e.g. '$3,000,000' -> 3000000.0."""
    if not text:
        return None
    match = re.search(r"\$?([\d,]+)", text)
    return float(match.group(1).replace(",", "")) if match else None

class RuleCheckInput(BaseModel):
    industry: str = Field(description="The business's primary industry")
    requested_limit: float | None = Field(default=None, description="Requested coverage limit in dollars")
    flood_zone: str | None = Field(default=None, description="FEMA flood zone letter, e.g. 'A' or 'V', if applicable")
    flood_mitigation_on_file: bool = Field(default=False, description="Whether flood mitigation documentation is on file")
    roof_age_years: int | None = Field(default=None, description="Age of the roof in years, if known")
    heavy_machinery: bool = Field(default=False, description="Whether the business operates heavy machinery (forklifts, industrial saws, commercial mowing equipment)")
    prior_liability_claims_5yr: int = Field(default=0, description="Number of liability claims in the past 5 years")

def check_underwriting_rules(
    industry: str,
    requested_limit: float | None = None,
    flood_zone: str | None = None,
    flood_mitigation_on_file: bool = False,
    roof_age_years: int | None = None,
    heavy_machinery: bool = False,
    prior_liability_claims_5yr: int = 0,
) -> dict:
    """Check known facts about a risk against explicit underwriting thresholds. No LLM call.

    Carried over from Session 3 (Exercise 2.1) with the flood-zone cap already filled in -
    this session isn't about writing new rules, it's about whether the agent supplies this
    function with the right arguments in the first place.
    """
    flags = []

    if requested_limit is not None and requested_limit > 5_000_000:
        flags.append("Requested limit exceeds standard $5,000,000 max - requires senior underwriter sign-off")

    if (
        flood_zone in ("A", "V")
        and requested_limit is not None
        and requested_limit > 2_000_000
        and not flood_mitigation_on_file
    ):
        flags.append(
            "Flood zone A/V with requested limit over $2,000,000 and no flood mitigation "
            "documentation on file - outside standard property appetite"
        )

    if heavy_machinery:
        flags.append("Heavy machinery present - supplemental safety questionnaire required before binding")

    if roof_age_years is not None and roof_age_years > 20:
        flags.append("Roof older than 20 years - current roof condition report required before binding")

    if prior_liability_claims_5yr >= 2:
        flags.append("2+ liability claims in past 5 years - outside preferred risk criteria")

    return {"flags": flags, "within_appetite": len(flags) == 0}

In [ ]:
class SearchDocumentsInput(BaseModel):
    query: str = Field(description="Natural-language question to search the underwriting guideline manuals for")
    k: int = Field(default=3, description="Number of matching excerpts to return")

class ExtractSubmissionInput(BaseModel):
    submission_text: str = Field(description="Full raw text of the underwriting submission to extract structured facts from")

TOOLS = [
    {
        "name": "search_documents",
        "description": (
            "Search the underwriting guideline manuals (general liability, property "
            "appetite, cyber exclusions, workers' comp classification) for text relevant "
            "to a natural-language question. Returns the top-k most relevant excerpts "
            "with their source document."
        ),
        "input_schema": SearchDocumentsInput.model_json_schema(),
    },
    {
        "name": "extract_submission_information",
        "description": (
            "Extract structured underwriting facts (business name, industry, locations, "
            "requested limits, notable risks, missing information) from the raw text of "
            "a new business submission."
        ),
        "input_schema": ExtractSubmissionInput.model_json_schema(),
    },
    {
        "name": "check_underwriting_rules",
        "description": (
            "Deterministically check known facts about a risk (requested limit, flood "
            "zone, roof age, heavy machinery, prior claims) against the underwriting "
            "manuals' explicit thresholds. Does not use an LLM - returns any rule "
            "violations found."
        ),
        "input_schema": RuleCheckInput.model_json_schema(),
    },
]

TOOL_FUNCTIONS = {
    "search_documents": search_documents,
    "extract_submission_information": extract_submission_information,
    "check_underwriting_rules": check_underwriting_rules,
}

for tool in TOOLS:
    print(tool["name"], "-", tool["description"][:60] + "...")

In [ ]:
import json

AGENT_SYSTEM_PROMPT = (
    "You are an underwriting assistant. You have three tools available: "
    "search_documents (search the underwriting guideline manuals), "
    "extract_submission_information (pull structured facts out of raw submission text), "
    "and check_underwriting_rules (check known facts against explicit underwriting thresholds). "
    "Given a new business submission, decide which tools you need and in what order. "
    "Gather enough information to determine whether the risk is within appetite before answering."
)

def run_agent_loop(user_message: str, system_prompt: str, tools: list[dict] = TOOLS, max_iterations: int = 6):
    response = client.generate_with_tools(user_message, tools, system_prompt=system_prompt)
    tool_call_log = []
    iterations = 0

    while response.tool_calls:
        if iterations >= max_iterations:
            break

        messages = response.messages
        for call in response.tool_calls:
            result = TOOL_FUNCTIONS[call.name](**call.arguments)
            tool_call_log.append({"name": call.name, "arguments": call.arguments, "result": result})
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

        response = client.generate_with_tools(user_message, tools, system_prompt=system_prompt, messages=messages)
        iterations += 1

    return response, tool_call_log

class UnderwritingDecision(BaseModel):
    """Final structured underwriting appetite decision."""
    decision: str = Field(description="One of: 'accept', 'decline', or 'refer to senior underwriter'")
    confidence: float = Field(description="Confidence in this decision, from 0.0 to 1.0")
    risk_factors: list[str] = Field(description="Specific risk factors identified from the gathered tool outputs")
    missing_information: list[str] = Field(description="Information that would be needed to be more certain")
    evidence: list[str] = Field(description="Specific tool outputs (guideline excerpts, rule flags, extracted facts) that support this decision")
    rationale: str = Field(description="Brief explanation tying the decision to the evidence above")

def run_full_agent(submission_text: str) -> tuple[UnderwritingDecision, list[dict]]:
    user_message = (
        f"Process this underwriting submission and determine whether it is within appetite:\n\n{submission_text}"
    )
    response, tool_call_log = run_agent_loop(user_message, AGENT_SYSTEM_PROMPT)

    tool_output_summary = "\n\n".join(
        f"Tool: {entry['name']}({entry['arguments']})\nResult: {entry['result']}"
        for entry in tool_call_log
    )
    decision_prompt = (
        f"Submission:\n{submission_text}\n\n"
        f"Tool outputs gathered while investigating this submission:\n{tool_output_summary}\n\n"
        "Using ONLY the information above, produce a final underwriting appetite decision. "
        "Every risk_factor and evidence entry must trace back to a specific tool output above - "
        "do not introduce new facts."
    )
    decision = client.generate_structured(decision_prompt, UnderwritingDecision, system_prompt=AGENT_SYSTEM_PROMPT)
    return decision, tool_call_log

In [ ]:
sample_submission = (
    "Meridian Cold Storage Partners operates a single refrigerated warehouse facility in "
    "Charleston, SC, located in a FEMA flood zone A. They are requesting $3,000,000 in "
    "property coverage for the building and ammonia-based refrigeration equipment. The "
    "roof was replaced 8 years ago. They have not provided any flood mitigation "
    "documentation. No liability claims have been filed in the past 5 years."
)

decision, tool_call_log = run_full_agent(sample_submission)
print(decision.model_dump_json(indent=2))

💬 **Reflection** — Read the decision above. It looks reasonable. But how do you actually know it's *correct*? What would you need to check to be sure — and could you check it again next week, after someone tweaks a prompt, without re-reading everything by eye?

---

## Section 2 — Define what good looks like

**Goal:** Before writing any evaluation code, agree on what "good" actually means for this agent's output — so the checks we build next measure something real.

Look again at the `UnderwritingDecision` above. There are at least four different questions you could ask about it, and they require completely different kinds of checks:

1. **Is the output well-formed?** Right fields, right types, `decision` is one of the three allowed values. This is a **structural** check — cheap, deterministic, catches nothing about whether the reasoning was any good.
2. **Is the output correct?** Does `decision` match what a human underwriter would conclude from this submission? Do `risk_factors` mention the things that actually matter? This is where you need either a known right answer (**deterministic correctness**, if you can define one) or a rubric-based judgment call (**LLM-as-judge**, if "correct" is more nuanced than an exact match).
3. **Is the output grounded?** Does every claim in `evidence` and `risk_factors` trace back to something a tool actually returned, or did the model state something it never looked up? This is a **quality** dimension a deterministic check can only partially catch (e.g. checking evidence strings appear in the tool call log) — it's a natural fit for an LLM judge too.
4. **Did the system behave reliably?** How long did it take, how many tokens did it use, how many tool calls, did anything error out or retry unnecessarily? This is **operational** — it doesn't tell you if any single answer was *correct*, but it tells you if the system is production-viable at all.

These four layers — **structural validation → deterministic correctness → model-based (judge) quality → operational reliability** — are what the rest of this notebook builds, one at a time. No single layer is enough on its own: a decision can be well-formed and still wrong, correct and still ungrounded, and correct-and-grounded while still being too slow or expensive to ship.

📝 **Exercise 2.1** — Before moving on, write down (a comment or a markdown cell is fine) 3 things about the decision above that a deterministic check *could* verify with certainty, and 1 thing about it that you don't think a simple deterministic check could ever settle on its own.

---

## Section 3 — Build a small evaluation dataset

**Goal:** Assemble a handful of submissions with known expected answers, so every check we write afterward has something concrete to check against.

An evaluation dataset doesn't need to be big — 4-6 well-chosen cases that cover distinct behaviors are worth more than 50 similar ones. Below are 4 pre-built cases; you'll fill in the expected answers for 2 more.

In [ ]:
# Case fields:
#   case_id                    - short identifier
#   submission                 - raw submission text, exactly as the agent would receive it
#   expected_decision          - one of "accept", "decline", "refer to senior underwriter"
#   expected_risk_factors      - phrases that MUST show up (fuzzy/substring match) somewhere
#                                 in the agent's risk_factors or evidence
#   expected_missing_information - phrases that MUST show up somewhere in missing_information
#   notes                      - why this case is in the dataset

EVAL_CASES = [
    {
        "case_id": "case_1_clear_accept",
        "submission": (
            "Fairview Stationery & Gifts operates a single retail storefront in Columbus, OH, "
            "selling stationery, greeting cards, and gift items. They are requesting $2,000,000 "
            "in general liability coverage. They have an active safety program on file, one "
            "liability claim in the past 5 years, and no history of policy cancellation."
        ),
        "expected_decision": "accept",
        "expected_risk_factors": [],
        "expected_missing_information": [],
        "notes": "Clean, unambiguous retail risk, well within standard GL appetite - nothing should be flagged.",
    },
    {
        "case_id": "case_2_clear_decline",
        "submission": (
            "Ridgeline Blasting & Demolition Co. is requesting general liability coverage for "
            "a structural demolition and controlled blasting operation on a construction site "
            "in Denver, CO. The scope of work includes explosive charge placement for building "
            "teardown."
        ),
        "expected_decision": "decline",
        "expected_risk_factors": ["explosives", "demolition"],
        "expected_missing_information": [],
        "notes": "Explosives/demolition operations are explicitly outside standard GL appetite per GL-GUIDELINES.",
    },
    {
        "case_id": "case_3_refer_missing_info",
        "submission": (
            "Blue Harbor Marina Cafe is a waterfront cafe and restaurant requesting $1,500,000 "
            "in property coverage for their building. The submission does not state a FEMA "
            "flood zone, roof age, construction type, or any prior loss history."
        ),
        "expected_decision": "refer to senior underwriter",
        "expected_risk_factors": [],
        "expected_missing_information": ["flood zone", "roof age", "construction type", "prior loss"],
        "notes": "Waterfront property with no flood/roof/construction data - not enough to assess appetite thresholds.",
    },
    {
        "case_id": "case_4_retrieval_dependent",
        "submission": (
            "Summit City Sports Apparel operates a large sporting-goods retail store, but "
            "approximately 15% of annual revenue comes from organizing and sponsoring "
            "semi-professional regional sports tournaments, including paying prize money to "
            "participants. They are requesting general liability coverage for the full "
            "operation."
        ),
        "expected_decision": "refer to senior underwriter",
        "expected_risk_factors": ["professional sports", "10 percent"],
        "expected_missing_information": [],
        "notes": (
            "check_underwriting_rules has no field for this - the only way to catch it is by "
            "retrieving GL-GUIDELINES' revenue-percentage rule for professional sports activities "
            "and reasoning over it. Tests retrieval-dependent correctness, not the deterministic tool."
        ),
    },
    {
        "case_id": "case_5_ambiguous",
        "submission": (
            "Coastal Bloom Florist is a small flower shop in a FEMA flood zone A coastal town, "
            "requesting $1,200,000 in property coverage. The building was constructed 15 years "
            "ago and the roof was replaced 5 years ago. No flood mitigation documentation is on "
            "file. No prior claims have been filed."
        ),
        "expected_decision": None,  # TODO (Exercise 3.1): fill this in
        "expected_risk_factors": None,  # TODO
        "expected_missing_information": None,  # TODO
        "notes": (
            "Ambiguous on purpose: it's in a flood zone with no mitigation on file, but the "
            "requested limit ($1.2M) is UNDER the $2,000,000 threshold that triggers the flood "
            "documentation requirement. Does the agent over-flag it as declined/referred just "
            "because of the flood zone, or correctly recognize the cap isn't triggered?"
        ),
    },
    {
        "case_id": "case_6_compound_rule",
        "submission": (
            "Tidewater Textile Warehouse is requesting $3,200,000 in property coverage for a "
            "42-year-old masonry warehouse. The property's FEMA Flood Insurance Rate Map "
            "designation is Zone VE (coastal high-hazard area subject to wave action). No "
            "flood mitigation documentation is on file. The roof was last replaced 23 years "
            "ago. Electrical systems were updated 4 years ago. No prior claims have been filed."
        ),
        "expected_decision": None,  # TODO (Exercise 3.1): fill this in
        "expected_risk_factors": None,  # TODO
        "expected_missing_information": None,  # TODO
        "notes": (
            "Intentionally-failing case: stacks TWO borderline PROPERTY-APPETITE conditions - "
            "a coastal flood zone over the $2,000,000 cap with no mitigation on file, AND a "
            "23-year-old roof. The flood zone is given as 'VE' (a FEMA coastal high-hazard "
            "subzone), not plain 'V' - check_underwriting_rules only recognizes the exact "
            "letters 'A' or 'V', so unless the agent normalizes 'VE' to 'V' before calling the "
            "tool, the flood/limit flag silently never fires, even though the roof-age flag "
            "still does. This is a real, verified failure mode (confirmed against the live "
            "agent, not assumed): the model tends to narrate the flood-zone concern in its "
            "final decision regardless, which makes it look 'caught' by eye - the only way to "
            "tell whether it was actually caught is to inspect check_underwriting_rules's own "
            "returned flags in the tool call log, not the narrative decision text."
        ),
    },
]

print(f"{len(EVAL_CASES)} cases loaded.")

📝 **Exercise 3.1** — Read `case_5_ambiguous` and `case_6_compound_rule` above (the `notes` field explains what each is testing) and fill in their `expected_decision`, `expected_risk_factors`, and `expected_missing_information` in the `EVAL_CASES` list above, then re-run that cell.

A few things to reason through:
- For `case_5_ambiguous`: is $1,200,000 above or below the $2,000,000 property flood-zone threshold from `check_underwriting_rules`? Given that, should this be an `accept`, or does the flood zone alone still warrant a `refer`?
- For `case_6_compound_rule`: what should `check_underwriting_rules` flag if it's called with the right arguments? Your `expected_risk_factors` should include phrases for **both** rule violations (flood/limit AND roof age) — that's what makes this check catch a *partial* fix, not just a total failure.

💬 **Reflection** — Why not reuse Session 3's crime/theft example (`off_topic_submission`) as one of these two cases? Because it was tested twice in Session 3 and the agent handled it correctly both times — a case that doesn't reliably fail can't be trusted to fail here either. An intentionally-failing case has to be verified to actually fail against the model you're running, not just assumed to.

💡 **Key Takeaway** — A good eval case tests one specific, nameable behavior — "does it apply this exact rule" or "does it recognize this exact gap" — not just "is the overall vibe right."

---

## Section 4 — Deterministic evaluations

**Goal:** Write cheap, exact-answer checks for the parts of the decision that have one unambiguous right answer.

These checks never call an LLM — they're plain Python comparing the agent's output against the `EVAL_CASES` you just built. They're fast, free, and 100% reproducible, which makes them the right tool for anything with a clear right answer. `validate_schema` and `check_decision` are provided as worked examples; you'll write `check_risk_factors` and `check_missing_information` yourself.

In [ ]:
def validate_schema(decision: UnderwritingDecision) -> dict:
    """Structural check: right type, decision is one of the 3 allowed values, confidence in range."""
    errors = []
    if decision.decision not in ("accept", "decline", "refer to senior underwriter"):
        errors.append(f"decision '{decision.decision}' is not one of the 3 allowed values")
    if not (0.0 <= decision.confidence <= 1.0):
        errors.append(f"confidence {decision.confidence} is outside the 0.0-1.0 range")
    if not isinstance(decision.risk_factors, list) or not isinstance(decision.evidence, list):
        errors.append("risk_factors/evidence are not lists")
    return {"passed": len(errors) == 0, "errors": errors}

def check_decision(decision: UnderwritingDecision, case: dict) -> dict:
    """Deterministic exact-match check against the case's expected_decision."""
    if case["expected_decision"] is None:
        return {"passed": None, "note": "expected_decision not filled in yet"}
    return {
        "passed": decision.decision == case["expected_decision"],
        "expected": case["expected_decision"],
        "actual": decision.decision,
    }

def check_tool_usage(tool_call_log: list[dict], case: dict) -> dict:
    """Worked example: flags obviously-broken tool behavior, not correctness of any one call -
    zero tool calls (agent guessed) or a suspiciously high count (looks like a retry loop).
    """
    if len(tool_call_log) == 0:
        return {"passed": False, "note": "agent produced a decision without calling any tools"}
    if len(tool_call_log) > 10:
        return {"passed": False, "note": f"{len(tool_call_log)} tool calls - looks like a retry loop"}
    return {"passed": True, "tools_called": [entry["name"] for entry in tool_call_log]}

📝 **Exercise 4.1** — Write `check_risk_factors` and `check_missing_information` below. Both compare a list of *expected phrases* against the agent's actual output — but an LLM will never phrase things exactly like your `expected_risk_factors` string. Use a **fuzzy/substring match** (case-insensitive `phrase in text`), not exact equality: for each expected phrase, check whether it appears in *any* of the candidate output strings. A helper, `_phrase_found`, is provided below to get you started.

In [ ]:
def _phrase_found(phrase: str, texts: list[str]) -> bool:
    """Case-insensitive substring match of `phrase` against a list of candidate strings."""
    phrase_lower = phrase.lower()
    return any(phrase_lower in text.lower() for text in texts)

def check_risk_factors(decision: UnderwritingDecision, case: dict) -> dict:
    """For each phrase in case['expected_risk_factors'], confirm it's found (fuzzy match)
    somewhere in decision.risk_factors or decision.evidence.
    """
    if case["expected_risk_factors"] is None:
        return {"passed": None, "note": "expected_risk_factors not filled in yet"}
    # TODO (Exercise 4.1): build the list of candidate texts to search (risk_factors + evidence),
    # then use _phrase_found to check each expected phrase. Return {"passed": bool, "missing": [...]}.
    raise NotImplementedError("Exercise 4.1: implement check_risk_factors")

def check_missing_information(decision: UnderwritingDecision, case: dict) -> dict:
    """For each phrase in case['expected_missing_information'], confirm it's found (fuzzy match)
    somewhere in decision.missing_information.
    """
    if case["expected_missing_information"] is None:
        return {"passed": None, "note": "expected_missing_information not filled in yet"}
    # TODO (Exercise 4.1): same pattern as check_risk_factors, but against decision.missing_information only.
    raise NotImplementedError("Exercise 4.1: implement check_missing_information")

In [ ]:
def run_deterministic_checks(decision: UnderwritingDecision, tool_call_log: list[dict], case: dict) -> dict:
    return {
        "schema": validate_schema(decision),
        "decision": check_decision(decision, case),
        "tool_usage": check_tool_usage(tool_call_log, case),
        "risk_factors": check_risk_factors(decision, case),
        "missing_information": check_missing_information(decision, case),
    }

# Run the agent once per case and keep the results around - Sections 5 and 6 reuse them
# instead of re-running the agent for every new kind of check.
AGENT_RESULTS = {}
for case in EVAL_CASES:
    decision, tool_call_log = run_full_agent(case["submission"])
    AGENT_RESULTS[case["case_id"]] = {"decision": decision, "tool_call_log": tool_call_log}
    print(f"Ran {case['case_id']}: decision={decision.decision!r}, {len(tool_call_log)} tool calls")

In [ ]:
for case in EVAL_CASES:
    result = AGENT_RESULTS[case["case_id"]]
    checks = run_deterministic_checks(result["decision"], result["tool_call_log"], case)
    print(f"\n=== {case['case_id']} ===")
    for check_name, outcome in checks.items():
        print(f"  {check_name}: {outcome}")

💬 **Reflection** — `case_6_compound_rule`'s `risk_factors` check may show `passed: True` even though something is actually broken — the model's narrative tends to mention the flood-zone concern regardless of whether `check_underwriting_rules` actually flagged it. A passing narrative-based check isn't proof the underlying tool call was correct. Look at the actual `check_underwriting_rules` call in `AGENT_RESULTS["case_6_compound_rule"]["tool_call_log"]` — what `flood_zone` argument did the agent pass, and does the tool's returned `flags` list actually include the flood/limit violation, or only the roof-age one? Keep this in mind; we'll come back to fix it in Section 6.

💡 **Key Takeaway** — Deterministic checks are cheap and exact, but only for things with one unambiguous right answer — and they're powerless against a well-formed decision that's wrong for reasons the check doesn't test.

---

## Section 5 — LLM as a judge

**Goal:** Build a judge for the things deterministic checks can't evaluate — groundedness and overall decision quality — using the exact same `generate_structured()` pattern from Session 1, just pointed at a rubric instead of an extraction schema.

An LLM judge is still just structured output: define a Pydantic scoring schema, write a system prompt describing how to score, call `generate_structured()`. The interesting part isn't the mechanism — it's the rubric.

### A first, weak attempt

Below is a deliberately underspecified rubric. Run it and look closely at what comes back.

In [ ]:
class JudgeScore(BaseModel):
    correctness: int = Field(description="1-5, is the decision correct")
    groundedness: int = Field(description="1-5, is the decision grounded in the evidence")
    completeness: int = Field(description="1-5, is anything important missing")
    decision_quality: int = Field(description="1-5, overall quality of the decision")
    passed: bool = Field(description="Whether this decision is good enough to ship")
    feedback: str = Field(description="Brief feedback")

JUDGE_SYSTEM_PROMPT_V1 = (
    "You are grading an underwriting decision. Score it on correctness, groundedness, "
    "completeness, and decision_quality from 1-5, and say whether it passed."
)

def run_judge_v1(submission_text: str, decision: UnderwritingDecision, tool_call_log: list[dict]) -> JudgeScore:
    prompt = (
        f"Submission: {submission_text}\n\n"
        f"Tool outputs: {tool_call_log}\n\n"
        f"Decision: {decision.model_dump_json()}"
    )
    return client.generate_structured(prompt, JudgeScore, system_prompt=JUDGE_SYSTEM_PROMPT_V1)

sample_result = AGENT_RESULTS["case_1_clear_accept"]
judge_v1_result = run_judge_v1(
    EVAL_CASES[0]["submission"], sample_result["decision"], sample_result["tool_call_log"]
)
print(judge_v1_result.model_dump_json(indent=2))

💬 **Reflection** — Look at the `feedback` field above. Does it cite anything specific from the tool outputs or the decision, or could that same feedback have been written without reading either one? A judge whose scores you can't trace back to specific evidence isn't much more trustworthy than reading the decision yourself.

This first rubric has real problems:
- No definition of what 1 vs. 5 *means* for any dimension — the model is free to invent its own scale
- No requirement to cite specific evidence, so `feedback` can be generic
- No guidance on how `passed` relates to the four scores (is 3/5 correctness a pass?)
- Nothing steering it away from grading its own kind of output favorably (the same family of model that produced the decision is grading it)

📝 **Exercise 5.1** — Rewrite the rubric as `JUDGE_SYSTEM_PROMPT_V2` below. At minimum, your version should:
1. Define what each score level (1 vs. 3 vs. 5) actually means for at least one dimension
2. Require `feedback` to quote or reference specific tool outputs / decision fields, not just describe them
3. State an explicit `passed` threshold (e.g. "passed requires groundedness >= 4 and correctness >= 4")

Then re-run the judge on the same case and compare `feedback` against the v1 result above.

In [ ]:
# TODO (Exercise 5.1): rewrite this rubric. Replace the placeholder text below with your own -
# be specific about what distinguishes a 1 from a 3 from a 5, require evidence citations in
# feedback, and state an explicit passing threshold.
JUDGE_SYSTEM_PROMPT_V2 = (
    "You are grading an underwriting decision for a commercial insurer. Score four dimensions "
    "from 1-5:\n"
    "- correctness: TODO - define what distinguishes 1 vs 3 vs 5 here\n"
    "- groundedness: TODO - e.g. 5 = every risk_factor/evidence entry traces to a specific tool "
    "output; 1 = claims not supported by any tool output\n"
    "- completeness: TODO\n"
    "- decision_quality: TODO\n"
    "In 'feedback', quote or directly reference specific tool outputs or decision fields - do not "
    "write generic praise or criticism.\n"
    "Set passed=true only if TODO: state your explicit threshold here, e.g. "
    "'groundedness >= 4 and correctness >= 4'."
)

def run_judge(submission_text: str, decision: UnderwritingDecision, tool_call_log: list[dict], system_prompt: str = JUDGE_SYSTEM_PROMPT_V2) -> JudgeScore:
    prompt = (
        f"Submission: {submission_text}\n\n"
        f"Tool outputs: {tool_call_log}\n\n"
        f"Decision: {decision.model_dump_json()}"
    )
    return client.generate_structured(prompt, JudgeScore, system_prompt=system_prompt)

judge_v2_result = run_judge(
    EVAL_CASES[0]["submission"], sample_result["decision"], sample_result["tool_call_log"]
)
print(judge_v2_result.model_dump_json(indent=2))

In [ ]:
JUDGE_RESULTS = {}
for case in EVAL_CASES:
    result = AGENT_RESULTS[case["case_id"]]
    JUDGE_RESULTS[case["case_id"]] = run_judge(case["submission"], result["decision"], result["tool_call_log"])

for case in EVAL_CASES:
    case_id = case["case_id"]
    deterministic = check_decision(AGENT_RESULTS[case_id]["decision"], case)
    judge = JUDGE_RESULTS[case_id]
    print(f"{case_id}: deterministic={deterministic.get('passed')}  judge.passed={judge.passed}  judge.groundedness={judge.groundedness}")

💬 **Reflection** — Where do the deterministic result and the judge's `passed` disagree, if anywhere? A disagreement isn't automatically a bug in one or the other — it might mean the judge caught a groundedness problem the deterministic check can't see, or it might mean the judge is being too lenient/strict. Which would you trust more for `case_2_clear_decline`, and which for the groundedness of `case_4_retrieval_dependent`'s reasoning?

💡 **Key Takeaway** — An LLM judge trades exactness for coverage — it can evaluate qualities no deterministic check can touch (groundedness, tone, completeness of reasoning), but it inherits every weakness of the model doing the judging: prompt sensitivity, inconsistency across runs, verbosity bias, and — since it's often the same model family that produced the answer — a tendency to grade its own kind of output generously. Use it for what deterministic checks can't do, not as a replacement for them.

---

## Section 6 — Observability, then improve and rerun

**Goal:** Capture the operational signal (latency, tokens, tool calls, errors) alongside correctness, then use everything built so far to diagnose and fix the one case we expect to be failing.

So far every check has asked "is the output right?" None of them ask "did the system behave reliably while producing it?" — how long it took, how many tokens it burned, how many tool calls it made, whether anything errored. That's the operational layer from Section 2, and it's what tells you whether a *correct* agent is actually viable to run in production.

One real limitation up front: `generate_structured()` (used for the final decision) doesn't return a `usage` object the way `generate_with_tools()` does — so the token counts captured below cover the agent's tool-calling turns only, not the final decision call. That's a genuine gap in what's observable through the current interface, not a bug in the wrapper below - worth flagging exactly the way you'd flag any other instrumentation gap in a real system.

In [ ]:
import time

def run_full_agent_observed(submission_text: str, tools: list[dict] = TOOLS, system_prompt: str = AGENT_SYSTEM_PROMPT) -> dict:
    """Same flow as run_full_agent, wrapped to also capture latency, token usage per
    tool-calling turn, and any error raised mid-run.
    """
    start = time.perf_counter()
    tool_call_log = []
    usage_log = []
    decision = None
    error = None

    try:
        user_message = (
            f"Process this underwriting submission and determine whether it is within appetite:\n\n{submission_text}"
        )
        response = client.generate_with_tools(user_message, tools, system_prompt=system_prompt)
        usage_log.append(response.usage)
        iterations = 0

        while response.tool_calls:
            if iterations >= 6:
                break
            messages = response.messages
            for call in response.tool_calls:
                result = TOOL_FUNCTIONS[call.name](**call.arguments)
                tool_call_log.append({"name": call.name, "arguments": call.arguments, "result": result})
                messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})
            response = client.generate_with_tools(user_message, tools, system_prompt=system_prompt, messages=messages)
            usage_log.append(response.usage)
            iterations += 1

        tool_output_summary = "\n\n".join(
            f"Tool: {entry['name']}({entry['arguments']})\nResult: {entry['result']}"
            for entry in tool_call_log
        )
        decision_prompt = (
            f"Submission:\n{submission_text}\n\n"
            f"Tool outputs gathered while investigating this submission:\n{tool_output_summary}\n\n"
            "Using ONLY the information above, produce a final underwriting appetite decision. "
            "Every risk_factor and evidence entry must trace back to a specific tool output above - "
            "do not introduce new facts."
        )
        decision = client.generate_structured(decision_prompt, UnderwritingDecision, system_prompt=system_prompt)
    except RuntimeError as e:
        error = str(e)

    retrieved_sources = [
        hit["source"]
        for entry in tool_call_log
        if entry["name"] == "search_documents"
        for hit in entry["result"]
    ]

    return {
        "decision": decision,
        "tool_call_log": tool_call_log,
        "elapsed_sec": round(time.perf_counter() - start, 2),
        "input_tokens": sum(u.input_tokens for u in usage_log),
        "output_tokens": sum(u.output_tokens for u in usage_log),
        "tool_call_count": len(tool_call_log),
        "retrieved_sources": retrieved_sources,
        "error": error,
    }

In [ ]:
def build_report(case: dict, observed: dict) -> dict:
    deterministic = run_deterministic_checks(observed["decision"], observed["tool_call_log"], case) if observed["decision"] else None
    judge = run_judge(case["submission"], observed["decision"], observed["tool_call_log"]) if observed["decision"] else None
    return {
        "case_id": case["case_id"],
        "decision": observed["decision"].decision if observed["decision"] else None,
        "deterministic_passed": deterministic["decision"].get("passed") if deterministic else None,
        "judge_passed": judge.passed if judge else None,
        "elapsed_sec": observed["elapsed_sec"],
        "input_tokens": observed["input_tokens"],
        "output_tokens": observed["output_tokens"],
        "tool_call_count": observed["tool_call_count"],
        "retrieved_sources": observed["retrieved_sources"],
        "error": observed["error"],
    }

REPORT = []
for case in EVAL_CASES:
    observed = run_full_agent_observed(case["submission"])
    REPORT.append(build_report(case, observed))

for row in REPORT:
    print(row)

### Diagnose the failing case

📝 **Exercise 6.1** — Find `case_6_compound_rule`'s row in `REPORT` above, and don't stop at `deterministic_passed` — as noted in Section 4, the narrative-based `risk_factors` check can pass even when something underneath is actually broken. Inspect its actual `check_underwriting_rules` call below: what `flood_zone` value did the agent pass, and does the tool's returned `flags` list include **both** the flood/limit violation and the roof-age violation, or only one?

In [ ]:
rule_check_calls = [
    entry for entry in AGENT_RESULTS["case_6_compound_rule"]["tool_call_log"]
    if entry["name"] == "check_underwriting_rules"
]
for call in rule_check_calls:
    print("arguments:", call["arguments"])
    print("result:", call["result"])

### Make a targeted fix

Look at the `flood_zone` argument the agent passed above compared to the submission text's `"Zone VE"` wording. `check_underwriting_rules`'s own rule only matches the exact letters `"A"` or `"V"` — if the agent passed `"VE"` unchanged, the flood/limit rule silently never fires, even though the roof-age rule still does. The root cause is the tool's **description**: nothing tells the model that FEMA flood zones have subzone suffixes (VE, AE, AH, AO, A99, etc.) that need to be normalized to their base letter before calling this tool — the same lesson from Session 3's Section 5 about tool descriptions steering behavior.

📝 **Exercise 6.2** — Edit `TOOLS_V2` below: rewrite `check_underwriting_rules`'s description to explicitly instruct the model to normalize FEMA flood zone subzone codes to their base letter (`"A"` or `"V"`) before passing `flood_zone`. Keep the change to the description only — don't touch `check_underwriting_rules`'s own code.

In [ ]:
TOOLS_V2 = [dict(t) for t in TOOLS]

# TODO (Exercise 6.2): replace this description with your own fix.
TOOLS_V2[2] = {
    **TOOLS_V2[2],
    "description": (
        "TODO - rewrite this. The current description doesn't tell the model that FEMA "
        "flood zone subzone codes (e.g. VE, AE, AH, AO, A99) need to be normalized to "
        "their base letter ('A' or 'V') before being passed as flood_zone, since this "
        "tool only recognizes the base letter."
    ),
}

case_6 = next(c for c in EVAL_CASES if c["case_id"] == "case_6_compound_rule")
observed_v2 = run_full_agent_observed(case_6["submission"], tools=TOOLS_V2, system_prompt=AGENT_SYSTEM_PROMPT)

rule_check_calls_v2 = [e for e in observed_v2["tool_call_log"] if e["name"] == "check_underwriting_rules"]
for call in rule_check_calls_v2:
    print("arguments:", call["arguments"])
    print("result:", call["result"])

print("\nDecision:", observed_v2["decision"].model_dump_json(indent=2) if observed_v2["decision"] else observed_v2["error"])

### Check for regressions

A fix that solves `case_6_compound_rule` but breaks something else isn't a fix. Rerun the whole dataset against `TOOLS_V2` and compare each case's deterministic result to its `REPORT` row from before the change.

In [ ]:
REPORT_V2 = []
for case in EVAL_CASES:
    observed = run_full_agent_observed(case["submission"], tools=TOOLS_V2, system_prompt=AGENT_SYSTEM_PROMPT)
    REPORT_V2.append(build_report(case, observed))

old_by_id = {row["case_id"]: row for row in REPORT}
for row in REPORT_V2:
    before = old_by_id[row["case_id"]]["deterministic_passed"]
    after = row["deterministic_passed"]
    changed = " <- CHANGED" if before != after else ""
    print(f"{row['case_id']}: before={before}  after={after}{changed}")

💬 **Reflection** — Did `case_6_compound_rule` flip from failing to passing? Did anything that was passing before flip to failing? A tool-description change is a global change — it affects every case that touches that tool, not just the one you were diagnosing, which is exactly why the regression check matters as much as the original fix.

💡 **Key Takeaway** — Observability doesn't tell you if an answer is correct, but it tells you where to look when something is wrong and whether a fix actually helped everywhere it touched, not just the one case you were staring at.

---

## Section 7 — Wrap-up

**Recap:**
- "Does it look right?" doesn't scale — a repeatable eval suite does, and it needs a real dataset with known expected answers to run against
- Four distinct layers, each catching something the others can't: **structural validation** (well-formed output) → **deterministic correctness** (exact-answer checks) → **LLM-as-judge** (groundedness, quality, anything with no single right answer) → **operational reliability** (latency, tokens, tool calls, errors)
- An LLM judge is only as good as its rubric — vague criteria produce vague, unverifiable feedback; explicit score definitions, evidence citations, and a stated pass threshold make it trustworthy enough to act on
- Observability doesn't grade correctness, but it's what lets you diagnose *why* something failed and confirm a fix didn't break anything else
- A "guaranteed failure" case has to be verified against the model you're actually running, not assumed — the same lesson Session 3 learned the hard way

💬 **Discussion** — Of the four layers you built today, which would you run on every single commit to this agent, and which would you only run occasionally (say, weekly, or before a release)? Consider cost, latency, and how often each layer's answer actually changes.

**What's next:** this notebook gives you a rerunnable answer to "did that change make the agent better or worse?" — the next step beyond a training lab would be wiring these checks into CI so that answer shows up automatically on every pull request, before a change ships.